# 05 · Dataset and performance of the two models

Report section 2.1 (dataset, Table 1) and section 3.1 (Table 3, Table 4,
Figure 3 and the numbers in Section 3.1.1).


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    P = Path("/content/drive/MyDrive/Affective Project/predictions")
else:
    P = Path("../predictions")

FILES = {
    "audio_v2_test": P / "audio_v2_test_predictions.csv",
    "audio_v2_validation": P / "audio_v2_validation_predictions.csv",
    "audio_v1_validation": P / "audio_v1_validation_predictions.csv",
    "audio_v2_history": P / "audio_v2_history.csv",
    "visual_test": P / "visual_test_predictions.csv",
    "visual_validation": P / "visual_validation_predictions.csv",
}
missing = [str(p) for p in FILES.values() if not p.exists()]
assert not missing, "Missing files:\n" + "\n".join(missing)
for k, p in FILES.items():
    print(f"{k:22s} {p}")

Mounted at /content/drive
audio_v2_test          /content/drive/MyDrive/Affective Project/predictions/audio_v2_test_predictions.csv
audio_v2_validation    /content/drive/MyDrive/Affective Project/predictions/audio_v2_validation_predictions.csv
audio_v1_validation    /content/drive/MyDrive/Affective Project/predictions/audio_v1_validation_predictions.csv
audio_v2_history       /content/drive/MyDrive/Affective Project/predictions/audio_v2_history.csv
visual_test            /content/drive/MyDrive/Affective Project/predictions/visual_test_predictions.csv
visual_validation      /content/drive/MyDrive/Affective Project/predictions/visual_validation_predictions.csv


In [2]:
D = Path("/content/drive/MyDrive/Affective Project/data") if IN_COLAB else Path("../data")
DATA = {"ratings": D / "summaryTable.csv", "actors": D / "VideoDemographics.csv"}
missing = [str(p) for p in DATA.values() if not p.exists()]
assert not missing, "Missing files:\n" + "\n".join(missing)

In [3]:
CLASSES = ["Anger", "Disgust", "Fear", "Happy", "Neutral", "Sad"]
PROB = [f"prob_{c}" for c in CLASSES]


def load_split(split):
    """One row per clip: true label and both models' predictions and probabilities.
    Clips without a visual prediction are dropped (only 1076_MTI_SAD_XX in validation)."""
    v = pd.read_csv(FILES[f"visual_{split}"])
    v = v[v["status"] == "ok"]
    a = pd.read_csv(FILES[f"audio_v2_{split}"])
    a["video_id"] = a["filename"].str.replace(".wav", "", regex=False)
    m = v[["video_id", "Speaker", "true_label", "predicted_label", *PROB]].merge(
        a[["video_id", "true_label", "predicted_label", *PROB]],
        on="video_id", suffixes=("_v", "_a"), validate="one_to_one")
    assert (m["true_label_v"] == m["true_label_a"]).all(), "label mismatch"
    m = m.rename(columns={"true_label_v": "y", "predicted_label_v": "pred_v",
                          "predicted_label_a": "pred_a"}).drop(columns="true_label_a")
    return m


def ece(p, y_idx, bins=10):
    """Expected calibration error with 10 equal-width confidence bins."""
    conf, pred = p.max(1), p.argmax(1)
    correct = (pred == y_idx).astype(float)
    edges = np.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        s = (conf > lo) & (conf <= hi)
        if s.any():
            total += s.mean() * abs(conf[s].mean() - correct[s].mean())
    return total

## Section 2.1: the CREMA-D dataset and the speaker split (Table 1)


In [4]:
ratings = pd.read_csv(DATA["ratings"], dtype=str)
actors = pd.read_csv(DATA["actors"])
clips = ratings.FileName.str.split("_", expand=True)
clips.columns = ["speaker", "sentence", "emotion", "intensity"]
clips["speaker"] = clips.speaker.astype(int)

print(f"Clips: {len(clips)}, actors: {len(actors)} ({actors.Sex.value_counts().to_dict()})")
print(f"Sentences: {clips.sentence.nunique()}")
print("Intensity levels per sentence:")
print(pd.crosstab(clips.sentence == "IEO", clips.intensity).rename(index={True: "IEO", False: "other 11"}))
print("\nShare of each emotion (%):", (100 * clips.emotion.value_counts(normalize=True)).round(1).to_dict())

Clips: 7442, actors: 91 ({'Male': 48, 'Female': 43})
Sentences: 12
Intensity levels per sentence:
intensity   HI   LO   MD    XX
sentence                      
other 11     0    0    0  5986
IEO        455  455  455    91

Share of each emotion (%): {'HAP': 17.1, 'SAD': 17.1, 'FEA': 17.1, 'ANG': 17.1, 'DIS': 17.1, 'NEU': 14.6}


In [5]:
test_spk = set(pd.read_csv(FILES["visual_test"]).Speaker)
val_spk = set(pd.read_csv(FILES["visual_validation"]).Speaker)
train_spk = set(actors.ActorID) - test_spk - val_spk
rows = []
for name, spk in [("Training", train_spk), ("Validation", val_spk), ("Test", test_spk),
                  ("Total", set(actors.ActorID))]:
    sex = actors[actors.ActorID.isin(spk)].Sex.value_counts()
    rows.append({"set": name, "clips": int(clips.speaker.isin(spk).sum()), "speakers": len(spk),
                 "men": int(sex.get("Male", 0)), "women": int(sex.get("Female", 0))})
print("Validation clips used after removing 1076_MTI_SAD_XX:", len(load_split("validation")))
pd.DataFrame(rows)

Validation clips used after removing 1076_MTI_SAD_XX: 1222


,set,clips,speakers,men,women
0,Training,4667,57,30,27
1,Validation,1223,15,8,7
2,Test,1552,19,10,9
3,Total,7442,91,48,43


In [6]:
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix

val = load_split("validation")
test = load_split("test")
print("Validation clips:", len(val), "| speakers:", val.Speaker.nunique())
print("Test clips:      ", len(test), "| speakers:", test.Speaker.nunique())

Validation clips: 1222 | speakers: 15
Test clips:       1552 | speakers: 19


## Table 3: validation and test results

In [7]:
rows = []
for name, col in [("Audio", "pred_a"), ("Visual", "pred_v")]:
    rows.append({"model": name,
                 "val_accuracy_%": 100 * accuracy_score(val.y, val[col]),
                 "val_macro_F1": f1_score(val.y, val[col], average="macro"),
                 "test_accuracy_%": 100 * accuracy_score(test.y, test[col]),
                 "test_macro_F1": f1_score(test.y, test[col], average="macro"),
                 "test_correct_clips": int((test.y == test[col]).sum())})
pd.DataFrame(rows).round(3)

,model,val_accuracy_%,val_macro_F1,test_accuracy_%,test_macro_F1,test_correct_clips
0,Audio,66.776,0.666,67.784,0.677,1052
1,Visual,64.566,0.647,68.041,0.673,1056


## Table 4: results for each emotion on the test set (%)

In [8]:
rows = []
for c in CLASSES:
    row = {"emotion": c, "n": int((test.y == c).sum())}
    for name, col in [("audio", "pred_a"), ("visual", "pred_v")]:
        p, r, f, _ = precision_recall_fscore_support(test.y, test[col], labels=[c], zero_division=0)
        row.update({f"{name}_P": 100 * p[0], f"{name}_R": 100 * r[0], f"{name}_F1": 100 * f[0]})
    rows.append(row)
pd.DataFrame(rows).round(1)

,emotion,n,audio_P,audio_R,audio_F1,visual_P,visual_R,visual_F1
0,Anger,265,74.0,76.2,75.1,70.6,63.4,66.8
1,Disgust,265,68.9,55.1,61.2,76.8,82.3,79.4
2,Fear,265,71.4,59.2,64.7,57.8,66.0,61.6
3,Happy,265,71.6,65.7,68.5,95.5,95.8,95.7
4,Neutral,227,65.1,87.2,74.6,53.9,63.9,58.5
5,Sad,265,58.3,66.0,61.9,50.0,36.2,42.0


## Figure 3: confusion matrices (% of each true emotion)

In [9]:
for name, col in [("Audio", "pred_a"), ("Visual", "pred_v")]:
    cm = confusion_matrix(test.y, test[col], labels=CLASSES)
    pct = pd.DataFrame(100 * cm / cm.sum(1, keepdims=True), CLASSES, CLASSES).round(0).astype(int)
    print(f"{name} model (rows = true emotion, columns = predicted emotion)")
    print(pct.to_string(), "\n")

Audio model (rows = true emotion, columns = predicted emotion)
         Anger  Disgust  Fear  Happy  Neutral  Sad
Anger       76        8     1      9        5    2
Disgust      9       55     4      5        5   22
Fear         5        3    59      9        8   17
Happy       12        5     8     66        7    3
Neutral      1        3     1      2       87    6
Sad          0        7    10      2       15   66 

Visual model (rows = true emotion, columns = predicted emotion)
         Anger  Disgust  Fear  Happy  Neutral  Sad
Anger       63        6    11      2       11    6
Disgust      0       82     9      2        0    6
Fear         5        4    66      0       12   12
Happy        1        0     2     96        0    1
Neutral     11        2    10      0       64   12
Sad         10       13    17      0       23   36 



## Section 3.1.1: the first audio model (v1) was overconfident

In [10]:
v1 = pd.read_csv(FILES["audio_v1_validation"])
P = v1[PROB].to_numpy()
y_idx = np.array([CLASSES.index(c) for c in v1.true_label])
print(f"v1 validation accuracy: {100 * v1.correct.mean():.1f}%")
print(f"v1 average confidence:  {100 * P.max(1).mean():.1f}%")
print(f"v1 ECE:                 {ece(P, y_idx):.2f}")

v1 validation accuracy: 58.8%
v1 average confidence:  86.0%
v1 ECE:                 0.27


## Section 3.1.1: training curve of the final audio model (Figure 2a, c)

In [11]:
h = pd.read_csv(FILES["audio_v2_history"])
best = h.loc[h.val_macro_f1.idxmax()]
print(f"Validation loss: epoch 1 = {h.val_loss.iloc[0]:.2f}, best epoch = {best.val_loss:.2f}")
print(f"Best epoch {int(best.epoch)}: validation accuracy {100 * best.val_accuracy:.1f}%, "
      f"macro-F1 {best.val_macro_f1:.3f}, training accuracy {100 * best.train_accuracy:.1f}%")

Validation loss: epoch 1 = 1.48, best epoch = 1.01
Best epoch 33: validation accuracy 66.8%, macro-F1 0.666, training accuracy 99.7%
